# 01 — Data Exploration & Analysis

**Goal:** Understand the existing dataset and identify data quality issues

**Outputs:**
- Dataset statistics (size, label distribution, text length)
- Data quality issues (duplicates, missing values, outliers)
- Vocabulary analysis (most common words, n-grams)
- Recommendations for preprocessing

In [1]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn wordcloud scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
import re

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

KeyboardInterrupt: 

## 1. Load Existing Datasets

In [ ]:
# Load your existing CSVs
df1 = pd.read_csv('../backend/training/fake_news_dataset_44k.csv')
df2 = pd.read_csv('../backend/training/fake_news_dataset_20k.csv')
df_true = pd.read_csv('../backend/training/True.csv')
df_fake = pd.read_csv('../backend/training/Fake.csv')

print(f"Dataset 1: {len(df1)} samples")
print(f"Dataset 2: {len(df2)} samples")
print(f"True.csv: {len(df_true)} samples")
print(f"Fake.csv: {len(df_fake)} samples")

## 2. Normalize & Merge

In [ ]:
# Normalize all to: {"text": "...", "label": 0/1}
# 0 = real, 1 = fake

def normalize_dataset(df, text_col, label_col):
    """Normalize to standard format."""
    normalized = pd.DataFrame()
    normalized['text'] = df[text_col].astype(str)
    
    # Convert labels to 0/1
    if df[label_col].dtype == 'object':
        normalized['label'] = df[label_col].map({'real': 0, 'fake': 1, 'true': 0, 'false': 1}).fillna(1)
    else:
        normalized['label'] = df[label_col]
    
    return normalized

# Merge all datasets
all_data = []

# Add your normalization logic here based on actual column names
# Example:
# all_data.append(normalize_dataset(df1, 'text', 'label'))

df_combined = pd.concat(all_data, ignore_index=True)
print(f"\nCombined dataset: {len(df_combined)} samples")
print(f"Label distribution:\n{df_combined['label'].value_counts()}")

## 3. Data Quality Analysis

In [ ]:
# Check for missing values
print("Missing values:")
print(df_combined.isnull().sum())

# Check for duplicates
duplicates = df_combined.duplicated(subset=['text']).sum()
print(f"\nDuplicate texts: {duplicates} ({duplicates/len(df_combined)*100:.1f}%)")

# Text length statistics
df_combined['text_length'] = df_combined['text'].str.len()
print(f"\nText length statistics:")
print(df_combined['text_length'].describe())

In [ ]:
# Visualize text length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df_combined['text_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Text Length Distribution')
axes[0].axvline(df_combined['text_length'].median(), color='red', linestyle='--', label='Median')
axes[0].legend()

# By label
for label in [0, 1]:
    subset = df_combined[df_combined['label'] == label]['text_length']
    axes[1].hist(subset, bins=50, alpha=0.6, label=f"{'Real' if label == 0 else 'Fake'}")
axes[1].set_xlabel('Text Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Text Length by Label')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Vocabulary Analysis

In [ ]:
# Word frequency analysis
def get_word_freq(texts, top_n=20):
    """Get most common words."""
    words = []
    for text in texts:
        words.extend(re.findall(r'\b\w+\b', text.lower()))
    return Counter(words).most_common(top_n)

real_words = get_word_freq(df_combined[df_combined['label'] == 0]['text'])
fake_words = get_word_freq(df_combined[df_combined['label'] == 1]['text'])

print("Top 20 words in REAL news:")
print(real_words)
print("\nTop 20 words in FAKE news:")
print(fake_words)

In [ ]:
# Word clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Real news word cloud
real_text = ' '.join(df_combined[df_combined['label'] == 0]['text'].head(1000))
wc_real = WordCloud(width=800, height=400, background_color='white').generate(real_text)
axes[0].imshow(wc_real, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Real News Word Cloud', fontsize=16)

# Fake news word cloud
fake_text = ' '.join(df_combined[df_combined['label'] == 1]['text'].head(1000))
wc_fake = WordCloud(width=800, height=400, background_color='white').generate(fake_text)
axes[1].imshow(wc_fake, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Fake News Word Cloud', fontsize=16)

plt.tight_layout()
plt.show()

## 5. Label Balance

In [ ]:
# Visualize label distribution
label_counts = df_combined['label'].value_counts()

plt.figure(figsize=(8, 6))
plt.bar(['Real (0)', 'Fake (1)'], label_counts.values, color=['green', 'red'], alpha=0.7)
plt.ylabel('Count')
plt.title('Label Distribution')
plt.text(0, label_counts[0] + 500, f"{label_counts[0]:,}", ha='center', fontsize=12)
plt.text(1, label_counts[1] + 500, f"{label_counts[1]:,}", ha='center', fontsize=12)
plt.show()

print(f"\nClass balance: {label_counts[0] / label_counts[1]:.2f}:1 (real:fake)")

## 6. Recommendations

Based on the analysis above, document:
1. Data quality issues found
2. Preprocessing steps needed
3. Class imbalance strategy (if needed)
4. Outliers to remove
5. Next steps for data collection

In [ ]:
# Save cleaned dataset
df_clean = df_combined.drop_duplicates(subset=['text']).copy()
df_clean = df_clean[df_clean['text_length'] >= 30]  # Min length filter
df_clean = df_clean[df_clean['text_length'] <= 5000]  # Max length filter

print(f"\nCleaned dataset: {len(df_clean)} samples (removed {len(df_combined) - len(df_clean)})")
df_clean[['text', 'label']].to_csv('../backend/training/dataset_clean.csv', index=False)
print("Saved to: backend/training/dataset_clean.csv")